# 1. Tissues-Specific Gene Expression

## 1.1 Reactome (Genes)

In [ ]:
from data.pathways.reactome import ReactomeNetwork

In [ ]:
R = ReactomeNetwork()

In [ ]:
hierarchy = R.reactome.hierarchy
human_hierarchy = hierarchy[hierarchy['child'].str.contains('HSA')]
human_hierarchy.to_csv("../Cytoscape_data/hierarchy.csv")
human_hierarchy

In [ ]:
pathway_genes.groupby('group').

In [ ]:
pathway_genes = R.reactome.pathway_genes
pathway_genes.to_csv("../Cytoscape_data/pathway_genes.csv")
pathway_genes

In [ ]:
pathway_names = R.reactome.pathway_names
human_pathway_names = pathway_names[pathway_names['reactome_id'].str.contains('HSA')]
human_pathway_names.to_csv("../Cytoscape_data/pathway_names.csv")

In [ ]:
import numpy as np
import pandas as pd 

In [ ]:
pathway_genes.columns = ['child', 'parent']

In [ ]:
edges = pd.concat([human_hierarchy,pathway_genes],axis=0)
edges.columns =['TARGET','SOURCE']
edges.to_csv('../Cytoscape_data/edges.csv')

In [ ]:
node = np.concatenate([pathway_genes.gene.unique(), human_pathway_names.reactome_id.unique()])

In [ ]:
edge

In [ ]:
reactome_gene = R.reactome.pathway_genes.gene.unique()

In [ ]:
len(reactome_gene)

In [ ]:
R.ge

## 1.2 FANTOM (Genes & Tisseus)

### 1.2.1 Read Data

In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np 

In [ ]:
fantom = pd.read_csv("_database/genes/FantomCAT_robust_genes_vs_traits_pvalue_nonzero_04072021.csv",header=1)

In [ ]:
fantom.head()

In [ ]:
fantom.describe()

In [ ]:
# value_counts of RNAs type
fantom_gene = fantom.geneName.values
fantom.geneClass.value_counts()

In [ ]:
fantom_ex = fantom.iloc[:, 10:] # fantom_ex = FANTOM gene EXpression matrix

fantom_ex_cod = fantom_ex[fantom.geneClass == "coding_mRNA"] # fantom_ex_cod = FANTOM gene EXpression in CODing rna 
fantom_ex_bin = fantom_ex < 0.05 # fantom_ex_bin = FANTOM gene EXpression in BINary if p-value > 0.05 
fantom_ex_log = -np.log(fantom_ex) # fantom_ex_log = FANTOM gene EXpression in LOGarithm 
fantom_ex_sig = fantom_ex[fantom_ex_bin.sum(axis=1) > 0] # fantom_ex_sig = FANTOM gene EXpression which are SIGnificant(p-values < 0.05)

fantom_ex_cod_bin = fantom_ex_cod < 0.05 # fantom_ex_cod_bin = FANTOM gene EXpression in CODing rna in BINary if p-value > 0.05 
fantom_ex_cod_log = -np.log(fantom_ex_cod) # fantom_ex_cod_log = FANTOM gene EXpression in CODing rna in LOGarithm 
fantom_ex_cod_sig = fantom_ex_cod[fantom_ex_cod_bin.sum(axis=1) > 0] # fantom_ex_cod_sig = FANTOM gene EXpression in CODing rna which are SIGnificant(p-values < 0.05)

fantom_ex_sig_bin = fantom_ex_sig < 0.05 # fantom_ex_bin_sig = FANTOM gene EXpression in BINary which are SIGnificant(p-values < 0.05)
fantom_ex_sig_log = -np.log(fantom_ex_sig) # fantom_ex_log_sig = FANTOM gene EXpression in LOGarithm  which are SIGnificant(p-values < 0.05)

fantom_ex_cod_sig_log = -np.log(fantom_ex_cod_sig)
fantom_ex_cod_sig_bin = fantom_ex_cod_sig < 0.05

In [ ]:
# plot of Number of tissues with p-values < 0.05
temp = fantom_ex_bin.sum(axis=1).value_counts().sort_index()
temp.plot(kind="bar",
          xlabel="Number of tissues with p-values < 0.05",
          ylabel="Number of genes")
plt.show()

In [ ]:
# plot of means of p-values in each gene distribution 
fantom_ex.mean(axis=1).hist(bins=50)

In [ ]:
# plot of standard deviation of p-values in each gene distribution 
fantom_ex.std(axis=1).hist(bins=50)

* Note: Means may not be a good choice because you may find some genes that have mean p-values < 1 or pre-defined threshold but are not significant(p-value<0.05)

In [ ]:
# there are 9960 genes with mean of p-values < 1 but are not significant
sum((fantom_ex.mean(axis=1)<1) * (fantom_ex_bin.sum(axis=1) == 0))

### 1.2.2 K-means clustering 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
num_clusters = 20

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(fantom_ex_bin.T)

In [ ]:
import matplotlib.pyplot as plt
 
# Create a range of values for k
k_range = range(1, 25)
 
# Initialize an empty list to
# store the inertia values for each k
inertia_values = []
 
# Fit and plot the data for each k value
for k in k_range:
    kmeans = KMeans(n_clusters=k, \
                    init='k-means++', random_state=42)
    y_kmeans = kmeans.fit_predict(fantom_ex.T)
    inertia_values.append(kmeans.inertia_)

# Plot the inertia values for each k
plt.plot(k_range, inertia_values, 'bo-')
plt.title('Elbow Method')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.show()

### 1.2.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# standardized_data = StandardScaler().fit_transform(fantom_ex_sig_bin.T)
pca = PCA(n_components=fantom_ex_sig_bin.shape[1]-1)
pca_data = pca.fit_transform(fantom_ex_sig_bin.T)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

### 1.2.4 TSNE

In [ ]:
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, verbose=1, perplexity=20, n_iter=500)
tsne_results = tsne.fit_transform(fantom_ex_bin.T)

In [ ]:
plt.scatter(tsne_results[:, 0], tsne_results[:, 1],c=clusters)
plt.xlabel("TSNE Component 1")
plt.ylabel("TSNE Component 2")
plt.title("TSNE Result")
plt.show()

### 1.2.5 Heatmap

In [ ]:
# Create the heatmap
plt.figure(figsize=(10, 6))
sns.clustermap(fantom_PCA_df.iloc[:50,:10], center=3, fmt=".5f", linewidths=0.5)
plt.title("Gene Expression Heatmap")
plt.xlabel("Samples")
plt.ylabel("Genes")
plt.show()

## 1.3 GTEx

### 1.3.1 Read Data 

In [ ]:
# download data from https://storage.googleapis.com/gtex_analysis_v8/rna_seq_data/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.gz
gtex = pd.read_csv("_database/genes/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct",sep="\t")

In [ ]:
gtex.head()

In [ ]:
gtex_gene = gtex.Description.values

In [ ]:
gtex.describe()

In [ ]:
# from column index=2 gene expression starts 
gtex.columns[:23]

In [ ]:
gtex_ex = gtex.iloc[:,2:]

### 1.3.2 K-means clustering

In [ ]:
from sklearn.cluster import KMeans
num_clusters = 10

kmeans = KMeans(n_clusters=num_clusters)
clusters = kmeans.fit_predict(gtex_ex.T)

### 1.3.3 PCA

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
standardized_data = StandardScaler().fit_transform(gtex_ex.T)
pca = PCA(n_components=standardized_data.shape[0]-1)
pca_data = pca.fit_transform(standardized_data)
np.cumsum(pca.explained_variance_ratio_)

In [ ]:
plt.scatter(pca_data[:, 0], pca_data[:, 1],c=clusters)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Result")
plt.show()

In [ ]:
fantom_PCA_df = pd.DataFrame(data=pca_data,index=fantom_ex.columns)

In [ ]:
fantom_PCA_df

In [ ]:
gtex_gene = gtex.Description.unique()
reactome_gtex_gene = np.intersect1d(reactome_gene, gtex_gene)
len(reactome_gtex_gene)

In [ ]:
# reactome and gtex intersection genes expression
reactome_gtex_index = [i in reactome_gtex_gene for i in gtex.Description]
reactome_gtex_df = gtex[reactome_gtex_index]

## 1.4 Comparison of Genes

In [ ]:
# import modules
from matplotlib_venn import venn3 
from matplotlib import pyplot as plt

In [ ]:
len(np.intersect1d(reactome_gene,gtex_gene))

In [ ]:
venn = venn3([set(reactome_gene), set(fantom_gene), set(gtex_gene)],
             set_labels=("Reactome", "Fantom","GTEx"))
plt.show()

# 2. Data 

In [1]:
import numpy as np
import importlib
from data.data_access import Data
from preprocessing import pre

params_file = 'train/params/P1000/pnet/onsplit_average_reg_10_tanh_large_testing.py'

In [2]:
loader = importlib.machinery.SourceFileLoader('params', params_file)
params = loader.load_module()

In [3]:
temp = Data(**params.data[0])

In [4]:
x, y, inf, col = temp.get_data()

In [5]:
x_train, x_validate_, x_test_, y_train, y_validate_, y_test_, info_train, info_validate_, info_test_, cols = temp.get_train_validate_test()

In [6]:
import pandas as pd 
tcga = pd.read_csv("_database/genes/tcga_prostate_expressed_genes_and_cancer_genes.csv")
hugo = pd.read_csv("_database/genes/HUGO_genes/protein-coding_gene_with_coordinate_minimal.txt",header=None,sep="\t")
hugo = hugo.iloc[:,3]
tcga = tcga.genes
from data.prostate_paper.data_reader import load_data_type
mut = load_data_type("mut_important", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
mut = mut[0]
print("mut.shape: ",mut.shape)
cnv_amp = load_data_type("cnv_amp", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_amp = cnv_amp[0]
print("cnv_amp.shape",cnv_amp.shape)
cnv_del = load_data_type("cnv_del", cnv_levels=3,
                     cnv_filter_single_event=True,
                     mut_binary=True, selected_genes=None)
cnv_del = cnv_del[0]
print("cnv_del.shape",cnv_del.shape)

mut_cnv = mut.columns.union(cnv_amp.columns)
mut_cnv_tcga = mut_cnv.intersection(tcga)
mut_cnv_tcga_hugo = mut_cnv_tcga.intersection(hugo)
len(mut_cnv_tcga_hugo)

mut.shape:  (1011, 14378)
cnv_amp.shape (1013, 13802)
cnv_del.shape (1013, 13802)


9229

In [7]:
X = np.concatenate((x_test_, x_validate_, x_train))
Y = np.concatenate((y_test_, y_validate_, y_train))

In [8]:
x_t = np.concatenate((x_test_, x_validate_))
y_t = np.concatenate((y_test_, y_validate_))
info_t = info_test_.append(info_validate_)

In [9]:
print("x_train:", x_train.shape)
print("x_train:", y_train.shape)

print("x_validate_:", x_validate_.shape)
print("y_validate_:", y_validate_.shape)

print("x_test:", x_t.shape)
print("y_test:", y_t.shape)

print("columns:", len(cols))

x_train: (807, 27687)
x_train: (807, 1)
x_validate_: (102, 27687)
y_validate_: (102, 1)
x_test: (204, 27687)
y_test: (204, 1)
columns: 27687


In [10]:
proc = pre.get_processor(params.pre)
if proc:
    proc.fit(x_train)
    x_train = proc.transform(x_train)
    x_test = proc.transform(x_t)
    if scipy.sparse.issparse(x_train):
        x_train = x_train.todense()
        x_test = x_test.todense()

{'type': None}


## Top-10 gene of PNET paper in prostate tissues from fantom 

In [ ]:
import pandas as pd 
fantom = pd.read_csv("_database/genes/FantomCAT_robust_genes_vs_traits_pvalue_nonzero_04072021.csv",header=1)

In [ ]:
list_gene = ['AR', 'TP53','PTEN', 'RB1','MDM4','FGFR1','MAML3','PDGFA','NOTCH1','EIF3E']
df = fantom.loc[fantom.geneName.isin(list_gene),:]
for i in df.columns:
    if i.lower().find('prostate') > -1:
        print(i)

In [ ]:
df.loc[:,'prostate gland']

In [ ]:
df.loc[:,'smooth muscle cell of prostate']

In [ ]:
df.loc[:,'prostate stromal cell']

In [ ]:
df.loc[:,'epithelial cell of prostate'] 

## Top-10 gene of PNET paper in prostate tissues from GTEx

In [ ]:
gtex = pd.read_csv("_database/genes/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct",sep="\t")

In [ ]:
df = gtex.loc[gtex.Description.isin(list_gene),:]

In [ ]:
for i in df.columns:
    if i.lower().find('prostate') > -1:
        print(i)

In [ ]:
df.loc[:,'Prostate']

In [ ]:
gtex.loc[:,'Prostate'].describe()

# 3. Model

In [11]:
from copy import deepcopy
from model import nn

## 3.1 Trainig 

* Note: Change the `single_outputs` argument in `onsplit_average_reg_10_tanh_large_testing.py` file to True to make single node output
        

In [14]:
model_params_ = deepcopy(params.models[0])

In [15]:
model = nn.Model(**model_params_['params'])

In [16]:
history = model.fit(x_train, y_train)

{'id': 'ALL', 'type': 'prostate_paper', 'params': {'data_type': ['mut_important', 'cnv_del', 'cnv_amp'], 'drop_AR': False, 'cnv_levels': 3, 'mut_binary': True, 'balanced_data': False, 'combine_type': 'union', 'use_coding_genes_only': True, 'selected_genes': 'tcga_prostate_expressed_genes_and_cancer_genes.csv', 'training_split': 0}}
n_hidden_layers 5


(1011, 27687)
(1011, 1)
(1011,)
(27687,)
input dimension 27687 self.units 9229
n_inputs_per_node 3.0
self.kernel_initializer <keras.src.regularizers.L2 object at 0x2a8e17ad0> <keras.src.initializers.initializers.LecunUniform object at 0x2a8e1dc10> <keras.src.regularizers.L2 object at 0x2a8e17ad0>
input dimensions (None, 27687)


2023-09-22 06:48:18.401400: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Max
2023-09-22 06:48:18.401427: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 64.00 GB
2023-09-22 06:48:18.401431: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 24.00 GB
2023-09-22 06:48:18.401499: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2023-09-22 06:48:18.401750: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


layer # 0
pathways 1387
genes 9275
filtered_map (9229, 0)
filtered_map (9229, 0)
filtered_map (9229, 0)
layer # 1
pathways 1066
genes 1399
filtered_map (1387, 0)
filtered_map (1387, 0)
filtered_map (1387, 0)
layer # 2
pathways 447
genes 1068
filtered_map (1066, 0)
filtered_map (1066, 0)
filtered_map (1066, 0)
layer # 3
pathways 147
genes 448
filtered_map (447, 0)
filtered_map (447, 0)
filtered_map (447, 0)
layer # 4
pathways 26
genes 147
filtered_map (147, 0)
filtered_map (147, 0)
filtered_map (147, 0)
layer # 5
pathways 1
genes 26
filtered_map (26, 0)
filtered_map (26, 0)
filtered_map (26, 0)
original dropout [0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
dropout [1, 2, 3, 4, 5] [0.5, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1] [0.001, 0.001, 0.001, 0.001, 0.001, 0.001, 0.001]
4749 15651.0
1387 1396.0
1066 1070.0
447 447.0
147 148.0
26 26.0
map is:
maps[0] 9229
maps[1] 1387
maps[2] 1066
maps[3] 447
maps[4] 147
maps[5] 26
layer 0, dropout  0.1 w_reg 0.001
layer 1, dropout  0.1 w_reg 0.001
layer 2, dropout  0.1

2023-09-22 06:48:25.928907: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


...

Epoch 255/300
9/9 - 1s - loss: 241.5113 - o1_loss: 0.1821 - o2_loss: 0.2701 - o3_loss: 0.2826 - o4_loss: 0.3038 - o5_loss: 0.3500 - o6_loss: 0.4108 - o1_f1: 0.9950 - o2_f1: 0.8246 - o3_f1: 0.8133 - o4_f1: 0.7994 - o5_f1: 0.7571 - o6_f1: 0.7430 - lr: 9.7656e-07 - 628ms/epoch - 70ms/step

Epoch 256: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 256/300
9/9 - 1s - loss: 241.5077 - o1_loss: 0.1821 - o2_loss: 0.2701 - o3_loss: 0.2826 - o4_loss: 0.3038 - o5_loss: 0.3500 - o6_loss: 0.4108 - o1_f1: 0.9945 - o2_f1: 0.8020 - o3_f1: 0.7890 - o4_f1: 0.7751 - o5_f1: 0.7678 - o6_f1: 0.7552 - lr: 9.7656e-07 - 610ms/epoch - 68ms/step

Epoch 257: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 257/300
9/9 - 1s - loss: 241.5041 - o1_loss: 0.1821 - o2_loss: 0.2701 - o3_loss: 0.2826 - o4_loss: 0.3038 - o5_loss: 0.3500 - o6_loss: 0.4108 - o1_f1: 0.9952 - o2_f1: 0.8252 - o3_f1: 0.8139 - o4_f1: 0.8001 - o5_f1: 0.7936 - o6_f1: 0.7792 - lr: 9.7656e-07 - 617ms/epoch - 69m


Epoch 278: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 278/300
9/9 - 1s - loss: 241.4278 - o1_loss: 0.1821 - o2_loss: 0.2701 - o3_loss: 0.2826 - o4_loss: 0.3037 - o5_loss: 0.3499 - o6_loss: 0.4106 - o1_f1: 0.9946 - o2_f1: 0.7669 - o3_f1: 0.7536 - o4_f1: 0.7424 - o5_f1: 0.7499 - o6_f1: 0.7739 - lr: 9.7656e-07 - 598ms/epoch - 66ms/step

Epoch 279: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 279/300
9/9 - 1s - loss: 241.4226 - o1_loss: 0.1821 - o2_loss: 0.2700 - o3_loss: 0.2826 - o4_loss: 0.3037 - o5_loss: 0.3499 - o6_loss: 0.4106 - o1_f1: 0.9940 - o2_f1: 0.8227 - o3_f1: 0.8121 - o4_f1: 0.8013 - o5_f1: 0.7913 - o6_f1: 0.7785 - lr: 9.7656e-07 - 595ms/epoch - 66ms/step

Epoch 280: LearningRateScheduler setting learning rate to 9.765625e-07.
Epoch 280/300
9/9 - 1s - loss: 241.4187 - o1_loss: 0.1821 - o2_loss: 0.2700 - o3_loss: 0.2826 - o4_loss: 0.3037 - o5_loss: 0.3499 - o6_loss: 0.4106 - o1_f1: 0.9950 - o2_f1: 0.8223 - o3_f1: 0.8099 - o4_f1: 0.

[]
['o1_loss', 'o2_loss', 'o3_loss', 'o4_loss', 'o5_loss', 'o6_loss']
['lr', 'o1_f1', 'o2_f1', 'o3_f1', 'o4_f1', 'o5_f1', 'o6_f1']
input dimensions (None, 27687)
21/26 [=======================>......] - ETA: 0s

2023-09-22 06:51:35.269562: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


26/26 [==============================] - 0s 6ms/step
(807, 1)
[0.1  0.11 0.12 0.13 0.14 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23
 0.24 0.25 0.26 0.27 0.28 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37
 0.38 0.39 0.4  0.41 0.42 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51
 0.52 0.53 0.54 0.55 0.56 0.57 0.58 0.59 0.6  0.61 0.62 0.63 0.64 0.65
 0.66 0.67 0.68 0.69 0.7  0.71 0.72 0.73 0.74 0.75 0.76 0.77 0.78 0.79
 0.8  0.81 0.82 0.83 0.84 0.85 0.86 0.87 0.88 0.89]
    accuracy  precision        f1    recall    th
0   0.510533   0.402421  0.573894  1.000000  0.10
1   0.524164   0.409231  0.580786  1.000000  0.11
2   0.553903   0.424920  0.596413  1.000000  0.12
3   0.596035   0.449324  0.620047  1.000000  0.13
4   0.660471   0.492593  0.660050  1.000000  0.14
..       ...        ...       ...       ...   ...
75  0.785626   1.000000  0.518106  0.349624  0.85
76  0.779430   1.000000  0.497175  0.330827  0.86
77  0.767038   1.000000  0.453488  0.293233  0.87
78  0.750929   1.000000  0.3927

In [17]:
test = model.predict(x_t)
res = y_t == test
res.sum()/len(y_t)

7/7 [==============================] - 0s 13ms/step
(204, 1)


0.8529411764705882

In [18]:
y_prob_6 = model.model.predict(x_t)

7/7 [==============================] - 0s 6ms/step


In [19]:
model.th

0.3699999999999999

In [20]:
for i in y_prob_6:
    res = y_t == (i>.3699999999999999)
    print(res.sum()/len(y_t))

0.7205882352941176
0.8627450980392157
0.8578431372549019
0.8333333333333334
0.8333333333333334
0.8480392156862745


## 3.2 Interpretability 

### 3.2.1 SHAP

#### single-output

In [21]:
import shap
import keras
import keras.backend as K
from keras.optimizers import Adam

In [22]:
trained_model = model.model

In [23]:
pathways = pd.read_csv('_database/pathways/Reactome2023/ReactomePathways.txt',sep='\t')
pathways.columns = ['reactome_id', 'pathway_name', 'species']
pathways = pathways[pathways.species == 'Homo sapiens'].reset_index(drop=True)
pathways.drop('species', inplace=True,axis=1)
name_counts = {}
new_names = []
for name in pathways['pathway_name']:
    if name in name_counts:
        name_counts[name] += 1
        new_name = f'{name} ({name_counts[name]})'
    else:
        name_counts[name] = 1
        new_name = name
    new_names.append(new_name)

# Update the 'description' column with unique descriptions
pathways['pathway_name'] = new_names



h1_features = pathways[pathways.reactome_id.isin(model.feature_names['h1'])]
h1_features = h1_features.set_index('reactome_id').loc[model.feature_names['h1']]['pathway_name'].tolist()


input_features = [f'{gene}_{status}' for gene, status in col]
h1_features = pathways.loc[pathways.reactome_id.isin(model.feature_names['h1']),'pathway_name'].tolist()
h2_features = pathways.loc[pathways.reactome_id.isin(model.feature_names['h2']),'pathway_name'].tolist()
h3_features = pathways.loc[pathways.reactome_id.isin(model.feature_names['h3']),'pathway_name'].tolist()
h4_features = pathways.loc[pathways.reactome_id.isin(model.feature_names['h4']),'pathway_name'].tolist()
h5_features = pathways.loc[pathways.reactome_id.isin(model.feature_names['h5']),'pathway_name'].tolist()


KeyError: "['R-HSA-1234162', 'R-HSA-211227', 'R-HSA-629602', 'R-HSA-191647', 'R-HSA-3359458', 'R-HSA-8853338', 'R-HSA-68827', 'R-HSA-69229', 'R-HSA-977441', 'R-HSA-977442', 'R-HSA-1300644', 'R-HSA-1300652', 'R-HSA-194840', 'R-HSA-2408517', 'R-HSA-375170', 'R-HSA-5661182', 'R-HSA-71182', 'R-HSA-109688', 'R-HSA-73777'] not in index"

In [ ]:
l = []
for i in model.feature_names['h1']:
    print(i)
    print(i,pathways.loc[pathways.reactome_id == i,'pathway_name'].item())
    l.append(pathways.loc[pathways.reactome_id == i,'pathway_name'].item())

In [ ]:
l = []
x = pathways[pathways.reactome_id == 'R-HSA-1234162']

In [ ]:
x

In [24]:
layer_name = [i.name for i in trained_model.layers]
layer_name

['inputs',
 'h0',
 'dropout_0',
 'h1',
 'dropout_1',
 'h2',
 'dropout_2',
 'h3',
 'dropout_3',
 'h4',
 'dropout_4',
 'h5',
 'o_linear0',
 'o_linear2',
 'o_linear3',
 'o_linear4',
 'o_linear5',
 'o_linear6',
 'o1',
 'o2',
 'o3',
 'o4',
 'o5',
 'o6']

In [25]:
trained_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 inputs (InputLayer)         [(None, 27687)]              0         []                            
                                                                                                  
 h0 (Diagonal)               (None, 9229)                 36916     ['inputs[0][0]']              
                                                                                                  
 dropout_0 (Dropout)         (None, 9229)                 0         ['h0[0][0]']                  
                                                                                                  
 h1 (SparseTF)               (None, 1387)                 17038     ['dropout_0[0][0]']           
                                                                                              

In [26]:
# how to make a model in functional api
inputs = trained_model.get_layer('inputs').output
o_linear0 = trained_model.get_layer('o_linear0')(inputs)
h0 = trained_model.get_layer('h0')(inputs)

o1 = trained_model.get_layer('o1')(o_linear0)
dropout_0 = trained_model.get_layer('dropout_0')(h0)

h1 = trained_model.get_layer('h1')(dropout_0)
o_linear2 = trained_model.get_layer('o_linear2')(h1)
o2 = trained_model.get_layer('o2')(o_linear2)

dropout_1 = trained_model.get_layer('dropout_1')(h1)
h2 = trained_model.get_layer('h2')(dropout_1)
o_linear3 = trained_model.get_layer('o_linear3')(h2)
o3 = trained_model.get_layer('o3')(o_linear3)

dropout_2 = trained_model.get_layer('dropout_2')(h2)
h3 = trained_model.get_layer('h3')(dropout_2)
o_linear4 = trained_model.get_layer('o_linear4')(h3)
o4 = trained_model.get_layer('o4')(o_linear4)

dropout_3 = trained_model.get_layer('dropout_3')(h3)
h4 = trained_model.get_layer('h4')(dropout_3)
o_linear5 = trained_model.get_layer('o_linear5')(h4)
o5 = trained_model.get_layer('o5')(o_linear5)

dropout_4 = trained_model.get_layer('dropout_4')(h4)
h5 = trained_model.get_layer('h5')(dropout_4)
o_linear6 = trained_model.get_layer('o_linear6')(h5)
o6 = trained_model.get_layer('o6')(o_linear6)

input dimensions (None, 27687)


In [ ]:
model_h1_output = keras.models.Model(name='model_h1_output', inputs=inputs, outputs=h1_as_input)
model_h2_output = keras.models.Model(name='model_h2_output', inputs=inputs, outputs=h2_as_input)
model_h3_output = keras.models.Model(name='model_h3_output', inputs=inputs, outputs=h3_as_input)
model_h4_output = keras.models.Model(name='model_h4_output', inputs=inputs, outputs=h4_as_input)
model_h5_output = keras.models.Model(name='model_h5_output', inputs=inputs, outputs=h5_as_input)

h1_output_tr = model_h1_output.predict(x_train)
h2_output_tr = model_h2_output.predict(x_train)
h3_output_tr = model_h3_output.predict(x_train)
h4_output_tr = model_h4_output.predict(x_train)
h5_output_tr = model_h5_output.predict(x_train)

h1_output_ts = model_h1_output.predict(x_t)
h2_output_ts = model_h2_output.predict(x_t)
h3_output_ts = model_h3_output.predict(x_t)
h4_output_ts = model_h4_output.predict(x_t)
h5_output_ts = model_h5_output.predict(x_t)

In [27]:
# how to make a model in functional api
inputs = trained_model.get_layer('inputs').output
o_linear0 = trained_model.get_layer('o_linear0')(inputs)
o1 = trained_model.get_layer('o1')(o_linear0)

h1_as_input = trained_model.get_layer('h1').output
o_linear2 = trained_model.get_layer('o_linear2')(h1_as_input)
o2 = trained_model.get_layer('o2')(o_linear2)

h2_as_input = trained_model.get_layer('h2').output
o_linear3 = trained_model.get_layer('o_linear3')(h2_as_input)
o3 = trained_model.get_layer('o3')(o_linear3)

h3_as_input = trained_model.get_layer('h3').output
o_linear4 = trained_model.get_layer('o_linear4')(h3_as_input)
o4 = trained_model.get_layer('o4')(o_linear4)

h4_as_input = trained_model.get_layer('h4').output
o_linear5 = trained_model.get_layer('o_linear5')(h4_as_input)
o5 = trained_model.get_layer('o5')(o_linear5)

h5_as_input = trained_model.get_layer('h5').output
o_linear6 = trained_model.get_layer('o_linear6')(h5_as_input)
o6 = trained_model.get_layer('o6')(o_linear6)


In [28]:
input_layers = [inputs,
                h1_as_input,
                h2_as_input,
                h3_as_input,
                h4_as_input,
                h5_as_input]

output_layers = [o1,
                 o2,
                 o3,
                 o4,
                 o5,
                 o6]

input_data = [x_train,
              h1_output_tr,
              h2_output_tr,
              h3_output_tr,
              h4_output_tr,
              h5_output_tr]


model_shap_input = keras.models.Model(name='model_shap_input',
                                inputs=input_layers[0],
                                outputs=output_layers[0])

model_shap_h1 = keras.models.Model(name='model_shap_h1',
                                inputs=input_layers[1],
                                outputs=output_layers[2])

model_shap_h2 = keras.models.Model(name='model_shap_h2',
                                inputs=input_layers[2],
                                outputs=output_layers[2])

model_shap_h3 = keras.models.Model(name='model_shap_h3',
                                inputs=input_layers[3],
                                outputs=output_layers[3])

model_shap_h4 = keras.models.Model(name='model_shap_h4',
                                inputs=input_layers[4],
                                outputs=output_layers[4])

model_shap_h5 = keras.models.Model(name='model_shap_h5',
                                inputs=input_layers[5],
                                outputs=output_layers[5])


NameError: name 'h1_output_tr' is not defined

In [ ]:
explainer = shap.DeepExplainer(model_shap_input, input_data[0])
shap_values_input = explainer.shap_values(input_data[0])

explainer = shap.DeepExplainer(model_shap_h1, input_data[1])
shap_values_h1 = explainer.shap_values(input_data[1])

explainer = shap.DeepExplainer(model_shap_h2, input_data[2])
shap_values_h2 = explainer.shap_values(input_data[2])

explainer = shap.DeepExplainer(model_shap_h3, input_data[3])
shap_values_h3 = explainer.shap_values(input_data[3])

explainer = shap.DeepExplainer(model_shap_h4, input_data[4])
shap_values_h4 = explainer.shap_values(input_data[4])

explainer = shap.DeepExplainer(model_shap_h5, input_data[5])
shap_values_h5 = explainer.shap_values(input_data[5])

In [ ]:
shap.summary_plot(shap_values_input[0], x_train, feature_names=input_features)

In [ ]:
len(h1_features)

In [ ]:
shap.summary_plot(shap_values_h1[0], input_data[1], feature_names=h1_features)

In [ ]:
shap.summary_plot(shap_values_h2[0], input_data[2])#, feature_names=h2_features)

In [ ]:
shap.summary_plot(shap_values_h3[0], input_data[3])#, feature_names=h3_features)

In [ ]:
shap.summary_plot(shap_values_h4[0], input_data[4])#, feature_names=h4_features)

In [ ]:
shap.summary_plot(shap_values_h5[0], input_data[5], feature_names=h5_features)

In [ ]:
shap_values_h5

#### multiple-output

In [ ]:
temp_model = keras.Model(name='temp_model', inputs=inputs, outputs=[o1,o2,o3,o4,o5,o6])

In [ ]:
def f1(y_true, y_pred):
    def recall(y_true, y_pred):
        """Recall metric.

        Only computes a batch-wise average of recall.

        Computes the recall, a metric for multi-label classification of
        how many relevant items are selected.
        """
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        possible_positives = K.sum(K.round(K.clip(y_true, 0, 1)))
        recall = true_positives / (possible_positives + K.epsilon())
        return recall

    def precision(y_true, y_pred):
        """Precision metric.

        Only computes a batch-wise average of precision.

        Computes the precision, a metric for multi-label classification of
        how many selected items are relevant.
        """
        true_positives = K.sum(K.round(K.clip(y_true * y_pred, 0, 1)))
        predicted_positives = K.sum(K.round(K.clip(y_pred, 0, 1)))
        precision = true_positives / (predicted_positives + K.epsilon())
        return precision

    precision = precision(y_true, y_pred)
    recall = recall(y_true, y_pred)
    return 2 * ((precision * recall) / (precision + recall + K.epsilon()))

In [ ]:
loss_weights = [2, 7, 20, 54, 148, 400]
optimizer = Adam(learning_rate=0.001)
temp_model.compile(optimizer=optimizer,
                   loss = ['binary_crossentropy'] * 6,
                   metrics=f1, 
                   loss_weights=loss_weights)
temp_model.fit(x_train, y_train,epochs=100)

#### check weights sanity 

In [ ]:
weights = modelll.get_weights()

In [ ]:
weights

In [ ]:
model_1.get_weights()

In [ ]:
model_2.get_weights()

### fake data

In [ ]:
from keras.layers import Input
from keras import Model

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Generate synthetic data with 10 features
num_samples = 1000
num_features = 10

# Generate random data for demonstration purposes
X_train = np.random.rand(num_samples, num_features)
y_train = np.random.rand(num_samples, 1)

# Create a simple feedforward neural network model
model = keras.Sequential([
    keras.layers.Dense(64, activation='relu', input_shape=(num_features,)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(1)  # Output layer with one neuron for regression
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
epochs = 50
batch_size = 32

model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size)

# You can now use this trained model to make predictions or save it for future use.


In [ ]:
X_train.shape

In [ ]:
model.summary()

In [ ]:

# Calculate SHAP values for a specific sample (e.g., the first sample)
explainer = shap.DeepExplainer(model, X_train)
shap_values = explainer.shap_values(X_train)


# Plot the SHAP values for the first sample (change the sample_idx as needed)
shap.summary_plot(shap_values, X_train, feature_names=[f"Feature {i+1}" for i in range(num_features)])


In [ ]:
shap_values

In [ ]:
# Example model_1
input_layer = keras.layers.Input(shape=(input_shape,))
hidden_layer = keras.layers.Dense(64, activation='relu')(input_layer)
output_layer = keras.layers.Dense(output_shape, activation='softmax')(hidden_layer)

model_1 = keras.Model(inputs=input_layer, outputs=output_layer)


## 3.3 Cross-Validation

In [ ]:
from sklearn.model_selection import KFold
from statistics import mean, stdev

In [ ]:
model_params_ = deepcopy(params.models[0])

In [ ]:
n_splits = 5  # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
accurasies = []
for train_indices, val_indices in kf.split(X):
    x_train_fold, x_val_fold = X[train_indices], X[val_indices]
    y_train_fold, y_val_fold = Y[train_indices], Y[val_indices]
    
    model = nn.Model(**model_params_['params'])
    
    model.fit(x_train_fold, y_train_fold)
    test = model.predict(x_val_fold)
    res = y_val_fold == test
    accurasies.append(res.sum()/len(y_val_fold))
    print("Accurasies: ", accurasies)

In [ ]:
accurasies

In [ ]:
print(mean(accurasies)) 
print(stdev(accurasies)) 
# Single Node : 0.8338243183924303 / Multiple Node : 0.8348241720723797
# Single Node : 0.014730575250953129 / Multiple Node : 0.033270867867846915